# Tutorial 01: Discover Pipeline

This notebook walks through the first two layers of pynpxpipe:
**Core infrastructure** (Session, config, logging) and **IO + Discover stage**.

By the end you will have:
- A fully initialized `Session` object
- `session.probes` populated with your recording's probe metadata
- Five visualizations of your data
- An understanding of how checkpoints enable resume-on-failure

**Prerequisites:** A SpikeGLX recording folder and a `.bhv2` behavioral file on disk.

---

In [ ]:
# === 用户配置区（修改这里）===========================================
# DATA_DIR: 根目录，需包含 SpikeGLX gate 文件夹（*_g0/ 等）和 .bhv2 文件
# OUTPUT_DIR: 处理结果输出目录（不存在会自动创建）
# SUBJECT_YAML: monkeys/*.yaml 中对应实验动物的配置文件

from pathlib import Path

DATA_DIR     = Path(r"C:\your\recording\root")   # <-- 修改这里
OUTPUT_DIR   = Path(r"C:\your\output")            # <-- 修改这里
SUBJECT_YAML = Path(r".\monkeys\YourMonkey.yaml") # <-- 修改这里
# =====================================================================

# Validate — friendly error messages instead of raw Python tracebacks
issues = []
if not DATA_DIR.exists():
    issues.append(f"❌ DATA_DIR not found: {DATA_DIR}")
if not SUBJECT_YAML.exists():
    issues.append(f"❌ SUBJECT_YAML not found: {SUBJECT_YAML}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if issues:
    for msg in issues:
        print(msg)
    raise SystemExit("请修改上方配置区中的路径，再重新运行此 cell。")

print("✅ 路径验证通过")
print(f"   DATA_DIR     = {DATA_DIR}")
print(f"   OUTPUT_DIR   = {OUTPUT_DIR}")
print(f"   SUBJECT_YAML = {SUBJECT_YAML}")

## Section 01: Session Setup

`Session` is pynpxpipe's central state object — a plain Python dataclass passed
through every stage. It holds paths, subject metadata, the probe list (populated
by DiscoverStage), and per-stage checkpoint status.

`SessionManager.from_data_dir()` auto-discovers the SpikeGLX gate folder
(`*_g0/` etc.) and the `.bhv2` file within `DATA_DIR`, then creates the output
directory structure and initializes logging.

**Design note:** `Session` has zero UI dependencies — no `click`, no `print`.
This means the same `Session` object works identically whether called from CLI,
this notebook, or a future GUI frontend.

In [ ]:
from pynpxpipe.core.config import load_subject_config
from pynpxpipe.core.session import Session, SessionManager
from pynpxpipe.core.logging import setup_logging

# 1. Load subject metadata from YAML
subject = load_subject_config(SUBJECT_YAML)
print(f"Subject loaded: {subject.subject_id} ({subject.species}, {subject.sex}, {subject.age})")

# 2. Create session — auto-discovers gate dir and .bhv2 inside DATA_DIR
#    Also creates: OUTPUT_DIR/checkpoints/, OUTPUT_DIR/logs/
session = SessionManager.from_data_dir(
    data_dir=DATA_DIR,
    subject=subject,
    output_dir=OUTPUT_DIR,
)
print(f"\nSession created:")
print(f"  session_dir : {session.session_dir}")
print(f"  output_dir  : {session.output_dir}")
print(f"  bhv_file    : {session.bhv_file}")
print(f"  probes      : {len(session.probes)} (populated after DiscoverStage)")

# 3. Setup structured logging (writes to OUTPUT_DIR/logs/*.log)
#    After this call, all stage output goes to the log file as JSON Lines.
setup_logging(session.log_path)
print(f"\nLogging to: {session.log_path}")